# EvidenceLab #03: Competing Risks
## When Another Event Gets There First
**Dr. Amobi Andrew Onovo | See it. Understand it. Run it.**

**Define the event → identify what can happen first → choose the estimand → then choose the model.**

Welcome. We will follow 2,500 simulated people from a common starting point, draw event-probability curves, and ask what changes when another event happens first. No upload or patient data is required. Choose **Runtime → Run all** in Google Colab. The first cell installs the analysis library. Allow a few minutes. Outputs are written to the runtime's `evidencelab03_outputs` folder; download that folder before the temporary runtime expires.

### Before the code: five useful words
* **Time-to-event data:** how long each person is observed, plus what happens at the end of observation.
* **Event of interest:** the precisely defined outcome whose timing we study.
* **Competing event:** an event that happens first and prevents that outcome from occurring later under the chosen endpoint definition.
* **Right-censoring:** observation ends while the event time is still unknown. This is not a third biological outcome.
* **Estimand:** the quantity our question asks us to estimate: for example, the probability of relapse by 60 months, before death.

Our teaching example is **first cancer relapse**, with **death before relapse** competing. Time zero is a hypothetical start of follow-up after treatment. These artificial data do not describe real cancer patients, HIV outcomes, or treatment effectiveness.

### The research story
During his PhD, Dr. Onovo analyzed survival among adults initiating ART in Nigeria. Professor Olivia Keiser asked: “What about the competing risks?” He learned the method, rebuilt the analysis in Stata, and incorporated it into work presented at IAS 2017 in Paris, 23–26 July 2017. Abstract **MOPEB0307**, printed page **79**, documents the historical analysis.

**Important distinction:** the abstract accounted for loss to follow-up using competing-risk regression. LTFU does not biologically prevent death. For all-cause mortality it may be missing outcome information or censoring, potentially informative. For a first recorded program outcome, a departure can be a competing endpoint; a multi-state formulation may be preferable for departures and returns. Define the question first.

## 1. Set up a reproducible workspace

**What this step does:** Install a pinned analysis library before importing it.

A documented version makes a lesson easier to reproduce.

**What to look for:** Installation finishes without an error; no cloud drive is mounted.

**Try changing:** Keep these versions for your first run. Use a fresh runtime if you have already imported incompatible packages.

In [ ]:
%pip -q install lifelines==0.30.3

NumPy generates repeatable random draws and works with arrays. pandas organizes tables. Matplotlib draws and exports figures. lifelines supplies Kaplan–Meier, Aalen–Johansen and Cox estimators. Standard-library modules record versions, dates and files. IPython displays lesson outputs.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from lifelines import KaplanMeierFitter, AalenJohansenFitter, CoxPHFitter
from lifelines.statistics import proportional_hazard_test
SEED = 201703
OUTPUT = Path("evidencelab03_outputs")
OUTPUT.mkdir(exist_ok=True)
plt.rcParams.update({"font.size": 13, "axes.spines.top": False, "axes.spines.right": False,
                     "figure.facecolor": "white", "axes.titleweight": "bold", "svg.fonttype": "none"})
COLORS = {0: "#687D92", 1: "#C82333", 2: "#008879", "km": "#1261CE"}

## 2. Specify the question and simulation

**What this step does:** Create a cohort with known event-generating mechanisms.

Simulation lets us change the competing-event rate while holding the primary mechanism fixed.

**What to look for:** The event is relapse; the competing event is death before relapse; follow-up ends at 60 months or independent early censoring.

**Try changing:** Later, change the competing multiplier. First run the baseline unchanged.

In [ ]:
PARAMETERS = {"n": 2500, "horizon_months": 60.0, "primary_rate": 0.009,
              "competing_rate": 0.010, "censor_rate": 0.0015,
              "competing_multiplier": 1.0}
display(pd.Series(PARAMETERS, name="Simulation setting").to_frame())

### Reusable simulation helper
Each person has age, a simplified binary sex variable, a randomly assigned exposure, and a measured baseline score. We generate two exponential waiting times with covariate-dependent rates and record whichever comes first. Independent random censoring and a 60-month administrative end can stop observation earlier. Exponential waiting times are a teaching device, not a claim that real hazards are constant. Latent clocks are conditionally independent **in this simulation**; the observed-data CIF estimator does not require independence between hypothetical latent event times.

The baseline score is measured, not an unobserved frailty. All predictors that determine the simulated primary hazard enter the Cox model. Sex coding is deliberately simplified and is not a statement about the diversity of real populations. Reusing the seed keeps participant covariates and primary waiting times identical across competing-event experiments.

In [ ]:
def simulate_cohort(n=2500, horizon_months=60, primary_rate=0.009,
                    competing_rate=0.010, censor_rate=0.0015,
                    competing_multiplier=1.0, seed=SEED):
    """Return observed first-event data; shared seed enables paired scenarios."""
    rng = np.random.default_rng(seed)
    age = np.clip(rng.normal(50, 12, n), 20, 85)
    sex_male = rng.binomial(1, 0.5, n)
    exposure = rng.binomial(1, 0.5, n)
    baseline_score = rng.normal(0, 1, n)
    rate1 = primary_rate * np.exp(0.20*(age-50)/10 + 0.10*sex_male - 0.30*exposure + 0.35*baseline_score)
    rate2 = competing_rate * competing_multiplier * np.exp(0.35*(age-50)/10 + 0.10*sex_male + 0.15*baseline_score)
    clock1 = rng.exponential(1/rate1)
    clock2 = rng.exponential(1/rate2)
    censor = np.minimum(rng.exponential(1/censor_rate, n), horizon_months)
    followup = np.minimum(np.minimum(clock1, clock2), censor)
    event = np.where(censor <= np.minimum(clock1, clock2), 0, np.where(clock1 < clock2, 1, 2))
    return pd.DataFrame({"age": age, "sex_male": sex_male, "exposure": exposure,
        "baseline_score": baseline_score, "followup_months": followup, "event_type": event})

In [ ]:
data = simulate_cohort(**PARAMETERS)
assert data.shape == (2500, 6)
assert set(data.event_type) == {0, 1, 2}
assert data.followup_months.between(0, PARAMETERS["horizon_months"]).all()
data.to_csv(OUTPUT / "simulated_cohort.csv", index=False)
display(data.head().round(2))
print(f"{len(data):,} people; {data.shape[1]} columns. All records are simulated.")

## 3. Meet the data

**What this step does:** Read the dictionary, counts and follow-up summary.

An event code without a clear definition can lead to the wrong analysis.

**What to look for:** Censored people have no observed first event before observation ends.

**Try changing:** Change the administrative horizon and rerun from simulation to see the counts change.

In [ ]:
dictionary = pd.DataFrame([
    ["age", "Age in years at time zero (20–85)"],
    ["sex_male", "Simplified simulated indicator: 1 male, 0 female"],
    ["exposure", "Randomized simulated exposure: 1 yes, 0 no"],
    ["baseline_score", "Measured standardized baseline risk score"],
    ["followup_months", "Time to first event or censoring, in months"],
    ["event_type", "0 censored; 1 relapse; 2 death before relapse"]], columns=["Variable", "Definition"])
display(dictionary)
STATUS = {0: "Censored", 1: "Relapse", 2: "Death before relapse"}
counts = data.event_type.value_counts().reindex([0, 1, 2], fill_value=0)
display(pd.DataFrame({"Status": [STATUS[k] for k in counts.index], "Count": counts.values,
                      "Percent": (100*counts.values/len(data)).round(1)}))
display(data.followup_months.describe().to_frame("Observed follow-up (months)"))

### Reusable figure-export helper
Each figure is saved as a high-resolution PNG and an editable vector SVG. The JSON/CSV outputs preserve unrounded numbers. Figures in the release video must come from these outputs.

In [ ]:
def export_figure(fig, name):
    """Save the same plotted figure for notebook, video and repository reuse."""
    fig.savefig(OUTPUT / f"{name}.png", dpi=180, bbox_inches="tight")
    fig.savefig(OUTPUT / f"{name}.svg", bbox_inches="tight")
    plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh([STATUS[k] for k in counts.index], counts.values, color=[COLORS[k] for k in counts.index])
ax.bar_label(bars, padding=8, fmt="%d")
ax.set(xlabel="People (simulated)", title="How observation ends", xlim=(0, counts.max()*1.18))
ax.invert_yaxis()
fig.tight_layout()
export_figure(fig, "event_status")

## 4. See time and observation

**What this step does:** Plot follow-up lengths and twelve individual timelines.

Observation ending is different from an event occurring.

**What to look for:** Symbols show the recorded endpoint; the bar ends there. A large mass at 60 months reflects administrative censoring.

**Try changing:** Display a different twelve-person slice; do not infer population patterns from this tiny illustration.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(data.followup_months, bins=np.arange(0, 65, 5), color="#1261CE", edgecolor="white")
axes[0].set(xlabel="Observed follow-up (months)", ylabel="People", title="Follow-up is not the same for everyone")
for row, person in enumerate(data.head(12).itertuples()):
    axes[1].hlines(row, 0, person.followup_months, color=COLORS[person.event_type], lw=2)
    axes[1].plot(person.followup_months, row, marker={0:"|",1:"o",2:"X"}[person.event_type], color=COLORS[person.event_type], ms=8)
axes[1].set(xlabel="Months since time zero", ylabel="Illustrative person", title="First recorded event or censoring", xlim=(0, 62))
fig.tight_layout()
export_figure(fig, "followup_and_timelines")

## 5. Kaplan–Meier: name the outcome first

**What this step does:** Fit survival free of either event, then a relapse-only KM calculation that codes deaths as censored.

1 − KM with competing events censored is not the real-world relapse CIF. Kaplan–Meier itself is not generally invalid.

**What to look for:** Any-event survival answers “neither event yet”. The relapse-only curve uses a different construction.

**Try changing:** Compare 24 and 60 months after the CIF section.

In [ ]:
km_any = KaplanMeierFitter(label="Free of relapse and death").fit(data.followup_months, data.event_type > 0)
km_relapse = KaplanMeierFitter(label="KM: deaths coded as censored").fit(data.followup_months, data.event_type == 1)
fig, ax = plt.subplots(figsize=(11, 5))
km_any.plot_survival_function(ax=ax, ci_show=True, color="#00345C")
ax.set(xlabel="Months since time zero", ylabel="Probability of neither event", title="Event-free survival: neither relapse nor death", ylim=(0, 1))
export_figure(fig, "event_free_survival")

**Interpretation:** ordinary censoring requires that those remaining observed represent those censored for the target, possibly conditional on measured variables. For a cause-specific Cox fit, coding competitors as censored is a computational way to remove them from that cause-specific risk set; it does **not** mean we assume they could still experience the first event. Interpreting 1 − relapse-only KM as risk in a hypothetical world without death needs additional assumptions; it is not a causal result here.

## 6. Cumulative incidence: the probability question

**What this step does:** Use the Aalen–Johansen estimator for each event type.

The cumulative incidence function (CIF) estimates P(first event is type k by time t), accounting for competing events.

**What to look for:** Both CIFs start near zero, never decrease, and together equal the probability of either event.

**Try changing:** Set selected_time to 24 or 36; inspect uncertainty and the number still observed.

In [ ]:
aj_relapse = AalenJohansenFitter(calculate_variance=True, seed=SEED)
aj_death = AalenJohansenFitter(calculate_variance=True, seed=SEED)
aj_relapse.fit(data.followup_months, data.event_type, event_of_interest=1)
aj_death.fit(data.followup_months, data.event_type, event_of_interest=2)
selected_time = 60.0
summary = {"time_months": selected_time, "km_relapse": float(1-km_relapse.predict(selected_time)),
           "cif_relapse": float(aj_relapse.predict(selected_time)), "cif_death": float(aj_death.predict(selected_time)),
           "event_free": float(km_any.predict(selected_time)),
           "at_risk_just_before": int((data.followup_months >= selected_time).sum())}
display(pd.Series(summary).to_frame("Estimate"))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
grid = np.linspace(0, 60, 601)
ax.step(grid, 1-km_relapse.predict(grid), where="post", color=COLORS["km"], ls="--", lw=2.5, label="1 − KM (death coded as censored)")
aj_relapse.plot_cumulative_density(ax=ax, color=COLORS[1], ci_show=False, lw=2.8, label="CIF: relapse")
aj_death.plot_cumulative_density(ax=ax, color=COLORS[2], ci_show=False, lw=2.5, ls="-.", label="CIF: death before relapse")
ax.set(xlabel="Months since time zero", ylabel="Estimated probability", xlim=(0, 60), ylim=(0, 0.65), title="Same data. Different answers.")
ax.legend(loc="upper left", frameon=False)
fig.text(0.13, 0.01, f"At {selected_time:g} months: 1 − KM {summary['km_relapse']:.1%} | relapse CIF {summary['cif_relapse']:.1%} | death CIF {summary['cif_death']:.1%}", fontsize=13)
fig.tight_layout(rect=(0, .06, 1, 1))
export_figure(fig, "km_cif_comparison")

The dashed blue line is a **naive absolute-risk interpretation**, not a competing-risk CIF. Censoring deaths in that calculation generally gives a larger estimate than relapse CIF. This gap is not a comparison of predictive accuracy between two fitted regression models. The following shaded bands show **pointwise 95% confidence intervals**, not simultaneous confidence bands or prediction intervals for one person.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
aj_relapse.plot_cumulative_density(ax=ax, color=COLORS[1], label="Relapse CIF", ci_alpha=.15)
aj_death.plot_cumulative_density(ax=ax, color=COLORS[2], label="Death-before-relapse CIF", ci_alpha=.15)
ax.set(title="Cumulative incidence, with pointwise uncertainty", xlabel="Months since time zero", ylabel="Estimated probability", xlim=(0,60), ylim=(0,.65))
export_figure(fig, "cif_uncertainty")
display(pd.DataFrame({"Month": [0,12,24,36,48,60], "Still at risk just before": [(data.followup_months >= t).sum() for t in [0,12,24,36,48,60]]}))

## 7. Cause-specific Cox: a rate question

**What this step does:** Fit covariate associations with the relapse hazard among people still free of both events.

A hazard is an instantaneous rate, not a probability. A hazard ratio is not a risk ratio.

**What to look for:** Exposure HR compares the instantaneous relapse rate for exposed versus unexposed people, at the same modeled covariates.

**Try changing:** Try omitting baseline_score and consider model misspecification; restore it for the reference analysis.

In [ ]:
cox_data = data.assign(age_decades=(data.age-50)/10, relapse=(data.event_type==1).astype(int))[
    ["followup_months", "relapse", "age_decades", "sex_male", "exposure", "baseline_score"]]
cox = CoxPHFitter().fit(cox_data, duration_col="followup_months", event_col="relapse")
cox_table = cox.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]]
display(cox_table.round(3))
cox_table.to_csv(OUTPUT / "cause_specific_cox.csv")
hr = float(cox.hazard_ratios_["exposure"])
display(Markdown(f"**Read it:** exposure HR = **{hr:.2f}** for the relapse hazard among those still event-free, conditional on modeled predictors. This is **not** the probability of relapse, a risk ratio, or evidence about a real treatment."))

## 8. Check the Cox assumptions

**What this step does:** Inspect scaled Schoenfeld residuals over time and a proportional-hazards diagnostic.

The Cox coefficient assumes a hazard ratio constant over follow-up; functional form and independent censoring also matter.

**What to look for:** A systematic residual trend can challenge proportional hazards. A large p-value does not prove assumptions hold.

**Try changing:** Discuss nonlinearity, time-varying effects or stratification before selecting a remedy.

In [ ]:
ph_test = proportional_hazard_test(cox, cox_data, time_transform="rank")
display(ph_test.summary)
ph_test.summary.to_csv(OUTPUT / "cox_ph_diagnostic.csv")
residuals = cox.compute_residuals(cox_data, kind="scaled_schoenfeld")
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, name in zip(axes.flat, residuals.columns):
    time = cox_data.loc[residuals.index, "followup_months"]
    ordered = pd.DataFrame({"time": time, "residual": residuals[name]}).sort_values("time")
    ax.scatter(ordered.time, ordered.residual, s=7, alpha=.2, color="#1261CE")
    ax.plot(ordered.time, ordered.residual.rolling(80, center=True, min_periods=30).mean(), color="#C82333", lw=2)
    ax.axhline(0, color="gray", lw=1)
    ax.set(title=name, xlabel="Event time (months)", ylabel="Scaled residual")
fig.suptitle("Look for time trends; the red smoother is descriptive")
fig.tight_layout()
export_figure(fig, "cox_diagnostics")

**Decision:** interpret the diagnostic together with the plots and the study design, not as a pass/fail button. Our conditional simulation uses proportional hazards and linear predictors by construction. A finite sample can still flag a pattern by chance. The analysis requires correct event coding and suitable independent censoring; this test cannot verify those. For real data, investigate functional form, measurement, missingness and confounding. Cox associations alone do not establish causality.

## 9. Fine–Gray: a different association

**What this step does:** Explain subdistribution hazard regression without inventing a Python estimator.

Fine–Gray relates covariates to the subdistribution hazard underlying a particular CIF, rather than the cause-specific hazard among only event-free people.

**What to look for:** An sHR is neither a risk ratio nor an individual event probability. Fine–Gray is not universally better than Cox.

**Try changing:** State whether your own question concerns absolute probability, a cause-specific rate, or a regression association with cumulative incidence.

**Implementation decision (verified for this release):** lifelines 0.30.3 provides KM, Aalen–Johansen and Cox; its documented API does not provide a Fine–Gray fitter. This notebook intentionally executes those validated methods and keeps Fine–Gray conceptual. It does not relabel a Cox fit as Fine–Gray or fabricate subdistribution HRs. The established R package [`cmprsk::crr`](https://cran.r-project.org/web/packages/cmprsk/cmprsk.pdf) is a documented Fine–Gray implementation for further work, with its own assumptions and diagnostics. A cross-language extension would need separate runtime and validation.

Fine–Gray uses a modified mathematical risk set that retains people who experienced a competing event, with censoring weights where needed. That is **not** a claim that those people remain physically able to experience a first relapse after death. Under a correctly specified proportional subdistribution hazards model, the coefficient describes an association with the subdistribution hazard and hence with the modeled CIF. Predictions still require a baseline function and covariate values. Check proportional subdistribution hazards and censoring assumptions. Cause-specific models for all causes can also be combined to predict CIFs.

In [ ]:
comparison = pd.DataFrame([
    ["Kaplan–Meier", "Survival for a defined endpoint", "How long until the endpoint?", "Any-event survival; suitable ordinary censoring"],
    ["Aalen–Johansen CIF", "Absolute probability by event type", "What proportion relapse before death by t?", "Competing-risk probability curves"],
    ["Cause-specific Cox", "Cause-specific hazard ratio", "How do covariates relate to rates while event-free?", "Conditional event-rate associations"],
    ["Fine–Gray (conceptual)", "Subdistribution hazard ratio", "How do covariates relate to the subdistribution hazard?", "CIF-related regression associations"]],
    columns=["Method", "Primary quantity", "Question", "Use"])
display(comparison)
comparison.to_csv(OUTPUT / "method_comparison.csv", index=False)

## 10. The experiment: change what can happen first

**What this step does:** Increase only the competing-event multiplier using paired random draws.

A different relapse CIF can arise without changing the relapse hazard mechanism.

**What to look for:** Higher competing-event frequency leaves fewer opportunities for a first relapse. These are model-based simulations, not intervention effects.

**Try changing:** Change multipliers (positive values only); leave primary_rate and seed fixed.

In [ ]:
scenario_curves, scenario_rows = {}, []
fig, ax = plt.subplots(figsize=(12, 6))
for label, multiplier, color, style in [("Low", .25, "#1261CE", "-"), ("Moderate", 1., "#C82333", "--"), ("High", 2.5, "#008879", "-.")]:
    settings = {**PARAMETERS, "competing_multiplier": multiplier}
    cohort = simulate_cohort(**settings)
    estimator = AalenJohansenFitter(calculate_variance=False, seed=SEED).fit(cohort.followup_months, cohort.event_type, event_of_interest=1)
    curve = np.asarray(estimator.predict(grid))
    scenario_curves[label] = curve.tolist()
    scenario_rows.append({"Scenario": label, "Competing multiplier": multiplier, "Relapse CIF at 60 months": float(estimator.predict(60))})
    ax.step(grid, curve, where="post", color=color, ls=style, lw=2.7, label=f"{label} competing-event rate")
ax.set(title="Same relapse mechanism. Different competing-event frequency.", xlabel="Months since time zero", ylabel="Cumulative incidence of relapse", xlim=(0,60), ylim=(0,.65))
ax.legend(frameon=False)
export_figure(fig, "competing_event_scenarios")
scenario_table = pd.DataFrame(scenario_rows)
display(scenario_table)
scenario_table.to_csv(OUTPUT / "scenario_results.csv", index=False)

## 11. Sanity checks and transparent arithmetic

**What this step does:** Cross-check the estimated probabilities using a direct risk-set recursion.

Reproducibility includes catching wrong event codes or accidentally plotting survival as risk.

**What to look for:** At every time, S + CIF1 + CIF2 = 1; CIFs are monotone; relapse-only 1 − KM is at least relapse CIF.

**Try changing:** Read the small hand-calculated example; changing its event labels changes the target.

### Reusable independent arithmetic helper
At each observed time, count those at risk just before it (Y), first relapses (d1), and competing deaths (d2). Add S(previous) × dk/Y to CIFk, then multiply event-free survival by 1 − (d1+d2)/Y. Censoring removes people from later risk sets without adding an event. Events are processed before censoring at a tied time. The library remains the production estimator; this independently written helper checks its arithmetic, including ties.

In [ ]:
def risk_set_recursion(times, events):
    """Transparent Aalen–Johansen recursion with grouped tied times."""
    times, events = np.asarray(times), np.asarray(events)
    survival, cif1, cif2, rows = 1.0, 0.0, 0.0, []
    for time in np.unique(times):
        at_risk = np.sum(times >= time)
        d1 = np.sum((times == time) & (events == 1))
        d2 = np.sum((times == time) & (events == 2))
        cif1 += survival*d1/at_risk
        cif2 += survival*d2/at_risk
        survival *= 1-(d1+d2)/at_risk
        rows.append([time, survival, cif1, cif2])
    return np.asarray(rows)

In [ ]:
audit = risk_set_recursion(data.followup_months, data.event_type)
np.testing.assert_allclose(audit[:,2], aj_relapse.predict(audit[:,0]), atol=1e-10)
np.testing.assert_allclose(audit[:,3], aj_death.predict(audit[:,0]), atol=1e-10)
np.testing.assert_allclose(audit[:,1]+audit[:,2]+audit[:,3], 1, atol=1e-10)
assert np.all(np.diff(audit[:,2:], axis=0) >= -1e-12)
assert np.all(1-np.asarray(km_relapse.predict(audit[:,0])) >= audit[:,2]-1e-12)
toy = risk_set_recursion([1, 2, 3, 4], [1, 2, 0, 1])
np.testing.assert_allclose(toy[-1,1:], [0, .75, .25])
print("PASS: independent risk-set arithmetic, probability identity, monotonicity, KM/CIF ordering and hand example.")

## 12. Cross-sector practice

**What this step does:** Name event, competitor and censoring before looking at the answers.

The same mathematics applies when mutually exclusive first outcomes are carefully defined.

**What to look for:** Observation ending is not automatically a competing event.

**Try changing:** Replace one scenario with your own research or program question.

| Setting | Your task |
|---|---|
| Health | After cancer treatment, study first relapse. Some people die without relapse; others are event-free when data collection ends. |
| Employment | Study first voluntary resignation from an employer. Some employees retire or are dismissed first; others still work there at study end. |
| Customer analytics | Study first account closure attributed to price. Others close for a mutually exclusive reason first; remaining accounts are still active at study end. |
| Engineering | Study first degradation failure of a component. Some components fail catastrophically first; others are functioning when testing ends. |

<details><summary>Reveal the answers</summary>

* Health: relapse / death before relapse / event-free study end.
* Employment: resignation / retirement or dismissal from this job / employed study end. If re-employment and repeated exits matter, use a multi-state or recurrent-event formulation.
* Customers: price-attributed first closure / another exclusive first closure / active study end. Ambiguous or multiple reasons require a better endpoint definition.
* Engineering: degradation failure / catastrophic first failure / functioning test end. Replacement and repair introduce new states.

In each row the order is **event / competitor / censoring**. Lost observation may be informative and needs additional analysis.
</details>

**Different industries. Same statistical problem: multiple possible events, and the first event can change what remains possible.**

## 13. Decision guide
1. Is the outcome time to an event? Define time zero, eligibility and follow-up.
2. Can more than one first event type occur? Define mutually exclusive categories.
3. Does one prevent the chosen event later, or does observation merely stop?
4. Is the target an absolute probability, an instantaneous rate, or a covariate association with cumulative incidence?
5. Choose the matching estimator/model and assess its assumptions.

**Probability by event type → CIF. Conditional event rate → cause-specific Cox. Subdistribution-hazard association → consider Fine–Gray. Neither event yet → any-event survival.**

For mortality after LTFU, ask what vital-status information is missing. Linkage, tracing, sensitivity analysis or multi-state modeling may be needed. Changing the code cannot resolve missing outcome knowledge.

## 14. Save the scientific record

**What this step does:** Export full-precision results, chart coordinates, data and environment metadata.

Every new number or chart in the video must trace back here.

**What to look for:** The output folder contains a claim ledger and reproducibility metadata; the seed and parameters are recorded.

**Try changing:** Download the outputs and rerun in a fresh runtime; numerical results should agree within floating-point tolerance.

In [ ]:
summary["exposure_cause_specific_hr"] = hr
summary["n"] = len(data)
metadata = {"seed": SEED, "parameters": PARAMETERS, "generated_utc": datetime.now(timezone.utc).isoformat(),
            "packages": {name: version(name) for name in ["numpy", "pandas", "scipy", "matplotlib", "lifelines"]},
            "fine_gray": "conceptual only; no fitted sHR", "data": "entirely simulated"}
curves = {"months": grid.tolist(), "km_relapse": (1-np.asarray(km_relapse.predict(grid))).tolist(),
          "cif_relapse": np.asarray(aj_relapse.predict(grid)).tolist(), "cif_death": np.asarray(aj_death.predict(grid)).tolist(),
          "scenarios": scenario_curves}
for name, value in [("results", summary), ("metadata", metadata), ("chart_data", curves)]:
    (OUTPUT / f"{name}.json").write_text(json.dumps(value, indent=2), encoding="utf-8")
ledger = pd.DataFrame([{"claim": key, "full_value": value, "source": "Notebook summary dictionary", "scope": "Simulated cohort; selected time where applicable"} for key, value in summary.items()])
ledger.to_csv(OUTPUT / "claim_ledger.csv", index=False)
display(metadata)
print("EVIDENCELAB03_RUN_ALL_PASS")

## 15. Take home
> **Define the event. Identify what can happen first. Choose the estimand. Then choose the model.**

> **Different models answer different questions.**

Try one experiment: increase the competing-event multiplier, rerun the notebook, and explain why relapse CIF changes even though the primary rate mechanism is unchanged.

### Sources and further study
* [IAS 2017 abstract book, MOPEB0307, printed page 79](https://www.ias2017.org/Portals/1/Files/IAS2017_LO.compressed4c6a.pdf?fileticket=m3LSDs1z4QY%3d&tabid=577&portalid=1).
* [lifelines Aalen–Johansen documentation](https://lifelines.readthedocs.io/en/latest/fitters/univariate/AalenJohansenFitter.html). Continuous event times avoid event ties here; administrative censoring ties remain valid.
* [lifelines Cox regression](https://lifelines.readthedocs.io/en/latest/Survival%20Regression.html) and [proportional hazards diagnostics](https://lifelines.readthedocs.io/en/latest/jupyter_notebooks/Proportional%20hazard%20assumption.html).
* [scikit-survival competing-risk estimator](https://scikit-survival.readthedocs.io/en/stable/api/generated/sksurv.nonparametric.cumulative_incidence_competing_risks.html), used in external independent validation.
* Fine JP, Gray RJ (1999). [A proportional hazards model for the subdistribution of a competing risk](https://doi.org/10.1080/01621459.1999.10474144). JASA 94:496–509.
* [CRAN cmprsk manual](https://cran.r-project.org/web/packages/cmprsk/cmprsk.pdf), validated R implementation for further study.

Educational simulation, not clinical advice or patient evidence. **EvidenceLab | See it. Understand it. Run it.**